Este notebook contém a modelagem do OHS utilizando PINNs.

In [9]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import joblib
import os

os.environ['CUDA_VISIBLE_DEVICES'] = ''
device = torch.device('cpu')
print(f"Dispositivo: {device}")

Dispositivo: cpu


In [10]:
df = pd.read_csv('C:/Users/User/OneDrive/Documentos/doutorado/PINN/oscilador-harmonico/data/04_feature/base_03/base_ohs.csv', sep=';', encoding='latin-1')
def converter_coluna(coluna):
    if coluna in df.columns:
        df[coluna] = df[coluna].astype(str).str.replace(',', '.').astype(float)

colunas_numericas = ['tempo', 'posicao', 'velocidade', 'x0', 'v0', 
                      'frequencia_angular', 'frequencia_linear', 'periodo_s',
                      'amplitude_max', 'energia_cinetica', 'energia_potencial', 'energia_mecanica']

for col in colunas_numericas:
    converter_coluna(col)

df.dtypes
df

,sistema_id,simulacao_id,id_trajetoria,tempo,posicao,velocidade,descricao_sistema,frequencia_angular,frequencia_linear,periodo_s,x0,v0,amplitude_max,energia_cinetica,energia_potencial,energia_mecanica
0,0,0,sistema_0_condicao_0,0.00,-0.125460,0.396323,Médio,5.0,0.795775,1.256637,-0.125460,0.396323,0.148373,0.078536,0.196752,0.275288
1,0,0,sistema_0_condicao_0,0.01,-0.121342,0.427180,Médio,5.0,0.795775,1.256637,-0.125460,0.396323,0.148373,0.091241,0.184047,0.275288
2,0,0,sistema_0_condicao_0,0.02,-0.116920,0.456969,Médio,5.0,0.795775,1.256637,-0.125460,0.396323,0.148373,0.104410,0.170878,0.275288
3,0,0,sistema_0_condicao_0,0.03,-0.112206,0.485616,Médio,5.0,0.795775,1.256637,-0.125460,0.396323,0.148373,0.117911,0.157377,0.275288
4,0,0,sistema_0_condicao_0,0.04,-0.107212,0.513049,Médio,5.0,0.795775,1.256637,-0.125460,0.396323,0.148373,0.131609,0.143679,0.275288
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63495,0,499,sistema_0_condicao_499,1.22,0.482010,0.336666,Médio,5.0,0.795775,1.256637,0.486211,-0.107988,0.486688,0.056672,2.904167,2.960839
63496,0,499,sistema_0_condicao_499,1.23,0.484773,0.215793,Médio,5.0,0.795775,1.256637,0.486211,-0.107988,0.486688,0.023283,2.937556,2.960839
63497,0,499,sistema_0_condicao_499,1.24,0.486324,0.094380,Médio,5.0,0.795775,1.256637,0.486211,-0.107988,0.486688,0.004454,2.956385,2.960839
63498,0,499,sistema_0_condicao_499,1.25,0.486659,-0.027268,Médio,5.0,0.795775,1.256637,0.486211,-0.107988,0.486688,0.000372,2.960467,2.960839


In [11]:
df['tempo'].max()

1.26

In [3]:
features_entrada = ['tempo', 'x0', 'v0', 'frequencia_angular']
features_saida = ['posicao', 'velocidade']

X_raw = df[features_entrada].values
y_raw = df[features_saida].values

print(f"Tamanho de X: {X_raw.shape}")
print(f"Tamanho de y: {y_raw.shape}")

scaler_X = StandardScaler()
X_normalized = scaler_X.fit_transform(X_raw)

scaler_y = StandardScaler()
y_normalized = scaler_y.fit_transform(y_raw)

n_samples = len(X_normalized)
indices = np.random.choice(len(X_normalized), n_samples, replace=False)
X_reduced = X_normalized[indices]
y_reduced = y_normalized[indices]

X_train, X_temp, y_train, y_temp = train_test_split(X_reduced, y_reduced, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Treino: {X_train.shape}, Validação: {X_val.shape}, Teste: {X_test.shape}")

joblib.dump(scaler_X, 'scaler_X.pkl')
joblib.dump(scaler_y, 'scaler_y.pkl')

Tamanho de X: (83400, 4)
Tamanho de y: (83400, 2)
Treino: (58380, 4), Validação: (12510, 4), Teste: (12510, 4)


['scaler_y.pkl']

In [4]:
class PINN(nn.Module):
    """
    Physics-Informed Neural Network para o Oscilador Harmônico Simples.
    """
    
    def __init__(self, input_dim=4, hidden_dim=128, output_dim=2, num_layers=4):
        super(PINN, self).__init__()
        
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.Tanh())
        
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())
        
        layers.append(nn.Linear(hidden_dim, output_dim))
        
        self.network = nn.Sequential(*layers)
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, t, x0, v0, omega):
        inputs = torch.cat([t, x0, v0, omega], dim=1)
        output = self.network(inputs)
        
        x_pred = output[:, 0:1]
        v_pred = output[:, 1:2]
        
        return x_pred, v_pred

In [5]:
class PINNLoss(nn.Module):
    """
    Função de custo: dados + física
    """
    def __init__(self, lambda_physics=0.1):
        super(PINNLoss, self).__init__()
        self.lambda_physics = lambda_physics
        self.mse_loss = nn.MSELoss()
    
    def forward(self, model, t, x0, v0, omega, x_true, v_true):
        if not t.requires_grad:
            t = t.clone().detach().requires_grad_(True)
        
        x_pred, v_pred = model(t, x0, v0, omega)
        
        # função de custo dos dados
        loss_data = self.mse_loss(x_pred, x_true) + self.mse_loss(v_pred, v_true)
        
        # função de custo física
        x_t = torch.autograd.grad(x_pred, t, grad_outputs=torch.ones_like(x_pred), 
                                   create_graph=True, retain_graph=True)[0]
        x_tt = torch.autograd.grad(x_t, t, grad_outputs=torch.ones_like(x_t), 
                                    create_graph=True, retain_graph=True)[0]
        
        physics_residual = x_tt + (omega ** 2) * x_pred
        loss_physics = torch.mean(physics_residual ** 2)
        
        loss_total = loss_data + self.lambda_physics * loss_physics
        
        return loss_total, loss_data, loss_physics

In [6]:
def prepare_batches(X, y, batch_size=256, device='cpu'):
    """
    Prepara batches para treinamento em CPU.
    """
    t = X[:, 0:1]
    x0 = X[:, 1:2]
    v0 = X[:, 2:3]
    omega = X[:, 3:4]
    
    x_true = y[:, 0:1]
    v_true = y[:, 1:2]
    
    t_tensor = torch.tensor(t, dtype=torch.float32, device=device, requires_grad=True)
    x0_tensor = torch.tensor(x0, dtype=torch.float32, device=device)
    v0_tensor = torch.tensor(v0, dtype=torch.float32, device=device)
    omega_tensor = torch.tensor(omega, dtype=torch.float32, device=device)
    x_true_tensor = torch.tensor(x_true, dtype=torch.float32, device=device)
    v_true_tensor = torch.tensor(v_true, dtype=torch.float32, device=device)
    
    dataset = TensorDataset(t_tensor, x0_tensor, v0_tensor, omega_tensor, 
                            x_true_tensor, v_true_tensor)
    
    return DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)


def train_pinn(model, train_loader, val_loader, epochs=100, lr=1e-3, device='cpu'):
    """
    Treina a PINN em CPU.
    """
    model = model.to(device)
    criterion = PINNLoss(lambda_physics=0.1)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)
    
    train_losses = []
    val_losses = []
    train_data_losses = []
    train_physics_losses = []
    
    for epoch in range(epochs):
        model.train()
        epoch_train_loss = 0
        epoch_train_data_loss = 0
        epoch_train_physics_loss = 0
        
        for t, x0, v0, omega, x_true, v_true in train_loader:
            optimizer.zero_grad()
            loss_total, loss_data, loss_physics = criterion(model, t, x0, v0, omega, x_true, v_true)
            loss_total.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            epoch_train_loss += loss_total.item()
            epoch_train_data_loss += loss_data.item()
            epoch_train_physics_loss += loss_physics.item()
        
        epoch_train_loss /= len(train_loader)
        epoch_train_data_loss /= len(train_loader)
        epoch_train_physics_loss /= len(train_loader)
        
        model.eval()
        epoch_val_loss = 0
        with torch.no_grad():
            for t, x0, v0, omega, x_true, v_true in val_loader:
                with torch.enable_grad():
                    loss_total, _, _ = criterion(model, t, x0, v0, omega, x_true, v_true)
                epoch_val_loss += loss_total.item()
        
        epoch_val_loss /= len(val_loader)
        
        train_losses.append(epoch_train_loss)
        val_losses.append(epoch_val_loss)
        train_data_losses.append(epoch_train_data_loss)
        train_physics_losses.append(epoch_train_physics_loss)
        
        scheduler.step(epoch_val_loss)
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch:4d} | Train Loss: {epoch_train_loss:.6f} | "
                  f"Data Loss: {epoch_train_data_loss:.6f} | Physics Loss: {epoch_train_physics_loss:.6f} | "
                  f"Val Loss: {epoch_val_loss:.6f}")
    
    return model, train_losses, val_losses, train_data_losses, train_physics_losses

In [7]:
print(f"\n=== CONFIGURAÇÃO DO CPU ===")
print(f"CUDA disponível: {torch.cuda.is_available()}")
print(f"Dispositivo: {device}")

BATCH_SIZE = 256     
HIDDEN_DIM = 128     
NUM_LAYERS = 4       
LEARNING_RATE = 1e-3
EPOCHS = 100       

print(f"\n=== CONFIGURAÇÕES DO MODELO ===")
print(f"Batch size: {BATCH_SIZE}")
print(f"Hidden dim: {HIDDEN_DIM}")
print(f"Num layers: {NUM_LAYERS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Epochs: {EPOCHS}")
print(f"Amostras de treino: {len(X_train):,}")
print(f"Batches por época: {max(1, len(X_train) // BATCH_SIZE)}")


=== CONFIGURAÇÃO DO CPU ===
CUDA disponível: False
Dispositivo: cpu

=== CONFIGURAÇÕES DO MODELO ===
Batch size: 256
Hidden dim: 128
Num layers: 4
Learning rate: 0.001
Epochs: 100
Amostras de treino: 58,380
Batches por época: 228


In [8]:
train_loader = prepare_batches(X_train, y_train, batch_size=BATCH_SIZE, device=device)
val_loader = prepare_batches(X_val, y_val, batch_size=BATCH_SIZE, device=device)

model = PINN(input_dim=4, hidden_dim=HIDDEN_DIM, output_dim=2, num_layers=NUM_LAYERS)

print(f"\n=== INICIANDO TREINAMENTO ===")

model, train_losses, val_losses, train_data_losses, train_physics_losses = train_pinn(
    model, train_loader, val_loader, epochs=EPOCHS, lr=LEARNING_RATE, device=device
)

print(f"\n=== TREINAMENTO CONCLUÍDO ===")


=== INICIANDO TREINAMENTO ===
Epoch    0 | Train Loss: 1.996039 | Data Loss: 1.991400 | Physics Loss: 0.046397 | Val Loss: 1.987355
Epoch   20 | Train Loss: 1.782889 | Data Loss: 1.778435 | Physics Loss: 0.044542 | Val Loss: 1.795361
Epoch   40 | Train Loss: 1.757414 | Data Loss: 1.751488 | Physics Loss: 0.059257 | Val Loss: 1.754956
Epoch   60 | Train Loss: 1.651471 | Data Loss: 1.644613 | Physics Loss: 0.068579 | Val Loss: 1.668509
Epoch   80 | Train Loss: 1.553165 | Data Loss: 1.545740 | Physics Loss: 0.074255 | Val Loss: 1.568119

=== TREINAMENTO CONCLUÍDO ===


In [9]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Evolução da Loss Total', 'Data Loss vs Physics Loss'),
    horizontal_spacing=0.15
)

# gráfico 1: Loss Total (Train e Validation)
fig.add_trace(
    go.Scatter(
        x=list(range(len(train_losses))),
        y=train_losses,
        mode='lines',
        name='Train Loss',
        line=dict(color='blue', width=2),
        hovertemplate='Epoch: %{x}<br>Train Loss: %{y:.6f}<extra></extra>'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=list(range(len(val_losses))),
        y=val_losses,
        mode='lines',
        name='Validation Loss',
        line=dict(color='red', width=2, dash='dash'),
        hovertemplate='Epoch: %{x}<br>Val Loss: %{y:.6f}<extra></extra>'
    ),
    row=1, col=1
)

# gráfico 2: Data Loss vs Physics Loss
fig.add_trace(
    go.Scatter(
        x=list(range(len(train_data_losses))),
        y=train_data_losses,
        mode='lines',
        name='Data Loss',
        line=dict(color='green', width=2),
        hovertemplate='Epoch: %{x}<br>Data Loss: %{y:.6f}<extra></extra>'
    ),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(
        x=list(range(len(train_physics_losses))),
        y=train_physics_losses,
        mode='lines',
        name='Physics Loss',
        line=dict(color='orange', width=2, dash='dash'),
        hovertemplate='Epoch: %{x}<br>Physics Loss: %{y:.6f}<extra></extra>'
    ),
    row=1, col=2
)

fig.update_layout(
    title=dict(
        text='Treinamento da PINN - Evolução das Funções de Custo',
        x=0.5,
        font=dict(size=16)
    ),
    width=1200,
    height=500,
    showlegend=True,
    legend=dict(
        x=0.02,
        y=0.98,
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1
    ),
    hovermode='closest'
)

fig.update_xaxes(
    title_text='Época',
    type='linear',
    row=1, col=1
)

fig.update_yaxes(
    title_text='Loss',
    type='log',
    row=1, col=1
)

fig.update_xaxes(
    title_text='Época',
    type='linear',
    row=1, col=2
)

fig.update_yaxes(
    title_text='Loss',
    type='log',
    row=1, col=2
)

fig.show()

In [10]:
torch.save({
    'model_state_dict': model.state_dict(),
    'scaler_X': scaler_X,
    'scaler_y': scaler_y,
    'config': {
        'input_dim': 4,
        'hidden_dim': HIDDEN_DIM,
        'output_dim': 2,
        'num_layers': NUM_LAYERS
    }
}, 'pinn_model_cpu.pth')

model.eval()
t_test = torch.tensor(X_test[:, 0:1], dtype=torch.float32, device=device)
x0_test = torch.tensor(X_test[:, 1:2], dtype=torch.float32, device=device)
v0_test = torch.tensor(X_test[:, 2:3], dtype=torch.float32, device=device)
omega_test = torch.tensor(X_test[:, 3:4], dtype=torch.float32, device=device)

with torch.no_grad():
    x_pred_norm, v_pred_norm = model(t_test, x0_test, v0_test, omega_test)
    
    x_pred = scaler_y.inverse_transform(np.column_stack([x_pred_norm.cpu().numpy(), v_pred_norm.cpu().numpy()]))[:, 0]
    v_pred = scaler_y.inverse_transform(np.column_stack([x_pred_norm.cpu().numpy(), v_pred_norm.cpu().numpy()]))[:, 1]
    
    x_true = scaler_y.inverse_transform(y_test)[:, 0]
    v_true = scaler_y.inverse_transform(y_test)[:, 1]

mse_x = mean_squared_error(x_true, x_pred)
mse_v = mean_squared_error(v_true, v_pred)
r2_x = r2_score(x_true, x_pred)
r2_v = r2_score(v_true, v_pred)

print(f"\n=== AVALIAÇÃO DO MODELO NOS DADOS DE TESTE ===")
print(f"MSE Posição: {mse_x:.6f}")
print(f"MSE Velocidade: {mse_v:.6f}")
print(f"R² Posição: {r2_x:.4f}")
print(f"R² Velocidade: {r2_v:.4f}")


=== AVALIAÇÃO DO MODELO NOS DADOS DE TESTE ===
MSE Posição: 5.501197
MSE Velocidade: 333.308085
R² Posição: 0.2227
R² Velocidade: 0.2904


In [ ]:
def evaluate_model(model, X_test, y_test, device='cpu'):
    """
    Avalia o modelo nos dados de teste.
    """
    model.eval()
    
    # prepara dados de teste
    t_test = torch.tensor(X_test[:, 0:1], dtype=torch.float32, device=device)
    x0_test = torch.tensor(X_test[:, 1:2], dtype=torch.float32, device=device)
    v0_test = torch.tensor(X_test[:, 2:3], dtype=torch.float32, device=device)
    omega_test = torch.tensor(X_test[:, 3:4], dtype=torch.float32, device=device)
    
    x_true = y_test[:, 0]
    v_true = y_test[:, 1]
    
    # previsões
    with torch.no_grad():
        x_pred, v_pred = model(t_test, x0_test, v0_test, omega_test)
        x_pred = x_pred.cpu().numpy().ravel()
        v_pred = v_pred.cpu().numpy().ravel()
    
    # métricas
        
    mse_x = mean_squared_error(x_true, x_pred)
    mse_v = mean_squared_error(v_true, v_pred)
    r2_x = r2_score(x_true, x_pred)
    r2_v = r2_score(v_true, v_pred)
    
    print("=" * 50)
    print("AVALIAÇÃO DO MODELO")
    print("=" * 50)
    print(f"MSE Posição: {mse_x:.6f}")
    print(f"MSE Velocidade: {mse_v:.6f}")
    print(f"R² Posição: {r2_x:.4f}")
    print(f"R² Velocidade: {r2_v:.4f}")
    
    return x_pred, v_pred, mse_x, mse_v

x_pred, v_pred, mse_x, mse_v = evaluate_model(model, X_test, y_test, device)

In [ ]:
# gráfico valores previstos vs reais
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test[:, 0], x_pred, alpha=0.3, s=1)
axes[0].plot([y_test[:, 0].min(), y_test[:, 0].max()], 
             [y_test[:, 0].min(), y_test[:, 0].max()], 'r--', linewidth=2)
axes[0].set_xlabel('Valor Real (Posição)')
axes[0].set_ylabel('Valor Previsto (Posição)')
axes[0].set_title(f'Posição - Previsões vs Real (R²={r2_score(y_test[:, 0], x_pred):.4f})')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(y_test[:, 1], v_pred, alpha=0.3, s=1)
axes[1].plot([y_test[:, 1].min(), y_test[:, 1].max()], 
             [y_test[:, 1].min(), y_test[:, 1].max()], 'r--', linewidth=2)
axes[1].set_xlabel('Valor Real (Velocidade)')
axes[1].set_ylabel('Valor Previsto (Velocidade)')
axes[1].set_title(f'Velocidade - Previsões vs Real (R²={r2_score(y_test[:, 1], v_pred):.4f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# selecionar uma condição específica para visualizar
# exemplo: sistema com x0=1, v0=0, ω=1
mask = (np.abs(X_test[:, 1] - 1.0) < 0.1) & (np.abs(X_test[:, 2]) < 0.1) & (np.abs(X_test[:, 3] - 1.0) < 0.1)

if np.any(mask):
    t_sample = X_test[mask, 0]
    x_true_sample = y_test[mask, 0]
    x_pred_sample = x_pred[mask]
    v_true_sample = y_test[mask, 1]
    v_pred_sample = v_pred[mask]
    
    # ordena por tempo
    idx_sorted = np.argsort(t_sample)
    t_sample = t_sample[idx_sorted]
    x_true_sample = x_true_sample[idx_sorted]
    x_pred_sample = x_pred_sample[idx_sorted]
    v_true_sample = v_true_sample[idx_sorted]
    v_pred_sample = v_pred_sample[idx_sorted]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Posição vs Tempo
    axes[0].plot(t_sample, x_true_sample, 'b-', label='Real', linewidth=2)
    axes[0].plot(t_sample, x_pred_sample, 'r--', label='PINN', linewidth=2)
    axes[0].set_xlabel('Tempo (s)')
    axes[0].set_ylabel('Posição (m)')
    axes[0].set_title('Posição vs Tempo')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Espaço de Fases
    axes[1].plot(x_true_sample, v_true_sample, 'b-', label='Real', linewidth=2)
    axes[1].plot(x_pred_sample, v_pred_sample, 'r--', label='PINN', linewidth=2)
    axes[1].set_xlabel('Posição (m)')
    axes[1].set_ylabel('Velocidade (m/s)')
    axes[1].set_title('Espaço de Fases')
    axes[1].legend()
    axes[1].grid(True, alpha=3)
    axes[1].axis('equal')
    
    plt.tight_layout()
    plt.show()